# 01 · Línea base facial (CondoHub · IA)

Embeddings congelados de FaceNet (MTCNN + InceptionResnetV1/VGGFace2) + SVM/k-NN. Evalúa con LFW; las fotos propias con consentimiento (I1) se suman aparte.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo → GPU (T4). Los datos y resultados se guardan en tu Google Drive, nunca en git (I6).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os

os.environ["ML_DATA_DIR"] = "/content/drive/MyDrive/condohub-ml"  # datasets y pesos (fuera de git)
os.makedirs(os.environ["ML_DATA_DIR"], exist_ok=True)

In [ ]:
# Código: clona el repo condohub-ia (cuando exista su URL) o copia la carpeta desde Drive.
REPO_URL = ""  # ej. "https://github.com/<org>/condohub-ia.git"
if REPO_URL:
    !git clone --depth 1 {REPO_URL} /content/condohub-ia
else:
    !cp -r "/content/drive/MyDrive/condohub-ml/codigo/condohub-ia" /content/condohub-ia
%cd /content/condohub-ia
!git log -1 --format=%h 2>/dev/null || echo "(sin git)"

In [ ]:
# facenet-pytorch fija versiones de torch que chocan con las de Colab: se instala sin sus dependencias.
!pip install -q facenet-pytorch --no-deps
!pip install -q scikit-learn pyyaml matplotlib pillow

In [ ]:
import json

import yaml

from ml.src import embeddings, lfw, linea_base, metricas
from ml.src.paths import dir_experimento

cfg = yaml.safe_load(open("ml/configs/facial_linea_base.yaml", encoding="utf-8"))
imagenes, etiquetas, nombres = lfw.cargar_lfw(cfg["dataset"]["min_fotos_por_persona"])
print(f"{len(etiquetas)} fotos de {len(nombres)} personas")

In [ ]:
embeber = embeddings.crear_embebedor()
emb, detectado = embeber(imagenes)
print(f"rostros detectados: {detectado.sum()} de {len(detectado)}")
emb, etiquetas = emb[detectado], etiquetas[detectado]  # se descartan las fotos sin rostro

In [ ]:
destino = dir_experimento(cfg["nombre"])
resultado = linea_base.ejecutar(None, etiquetas, lambda _: emb, cfg, destino)
print(json.dumps({k: resultado[k] for k in ("personas", "imagenes", "verificacion")}, indent=2))
print({m: round(v["exactitud"], 4) for m, v in resultado["identificacion"].items()})
print("guardado en", destino)

In [ ]:
# Curva ROC de verificación sobre la partición de PRUEBA (el umbral ya se calibró en validación).
import matplotlib.pyplot as plt

from ml.src.split import dividir_por_persona

idx = dividir_por_persona(etiquetas, tuple(cfg["proporciones"]), cfg["semilla"])["test"]
i, j, y = metricas.generar_pares(etiquetas[idx], cfg["pares_por_parte"], cfg["semilla"])
curva = metricas.curva_far_frr(metricas.similitud_coseno(emb[idx][i], emb[idx][j]), y)
plt.plot(curva.far, 1 - curva.frr)
plt.xlabel("FAR")
plt.ylabel("1 - FRR")
plt.title("ROC · prueba")
plt.grid(True)
plt.savefig(destino / "roc.png", dpi=120)
plt.show()

## Qué hacer con los resultados

1. Copia `metrics.json` y `roc.png` de la carpeta del experimento al repo (`ml/experiments/`): **sí se versionan**; los pesos y fotos, no.
2. Anota en `ml/README.md` el umbral elegido y las métricas (la regla I2 pide documentarlo ahí).
3. Un modelo ajustado (triplet loss) solo pasa al servicio si supera esta línea base.